In [20]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [21]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [22]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [23]:
article_text = """
Earth's atmosphere and oceans were formed through a combination of natural processes following Earth's formation.
Earth's atmosphere and oceans were formed by volcanic activity and outgassing. [1]
Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets. [2]
Sufficient water to fill the oceans may have been on Earth since it formed. [3] In this model, atmospheric greenhouse gases kept the oceans from freezing when the newly forming Sun had only 70% of its current luminosity. [4]
By 3.5 billion years ago, Earth's magnetic field was established, which helped prevent the atmosphere from being stripped away by the solar wind. [5] This magnetic field protection was crucial for maintaining the atmosphere that had developed.
The atmosphere that formed has a composition of primarily nitrogen and oxygen. [6] Water vapor is widely present in the atmosphere, forming clouds that cover most of the planet. [7] These elements combined to create the conditions necessary for life to develop on Earth."""

In [24]:
# TODO: Read image data, feed into Claude

with open("earth.pdf", "rb") as f:
    file_bytes = base64.b64encode(f.read()).decode('utf-8')

Messages = []

add_user_message(Messages, [
    {
        "type": "document",
        "source":{
            "type": "text",
            "media_type": "text/plain",
            "data": article_text,
        },
        "title": "Earth Article",
        "citations":{
            "enabled": True,
        }
    },
    {
        "type": "text",
        "text": "How were Earth's atmosphere and oceans were formed?",
    }
])

chat(Messages)

Message(id='msg_013smE26hvuNNFhKvGfsiji7', container=None, content=[TextBlock(citations=[CitationCharLocation(cited_text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing. ", document_index=0, document_title='Earth Article', end_char_index=194, file_id=None, start_char_index=115, type='char_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationCharLocation(cited_text='[1]\nWater vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets. ', document_index=0, document_title='Earth Article', end_char_index=325, file_id=None, start_char_index=194, type='char_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text'), TextBlock(citations=None, text='\n\n', type='te